<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex08.2-transient-heat/Ex08.2_05_compare_and_report.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_08.2 · Notebook 05 — Compare, and Report

**Paired with L8.2 · Dynamic Heat**

One equation, four models: a soft initial condition, a hard one, a domain with
a hole, and a diffusivity that was not given to you.

---

## 0 · Setup and what the other notebooks produced

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08.2-transient-heat/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
runs = {
    "01 · soft IC":       "nb01_soft.npz",
    "02 · hard IC":       "nb02_hard.npz",
    "03 · plate in time": "nb03_plate.npz",
    "04 · inverse alpha": "nb04_inverse.npz",
}
R = {}
for label, fn in runs.items():
    p = os.path.join("Ex08.2_outputs", fn)
    if os.path.exists(p):
        R[label] = np.load(p, allow_pickle=True)
        print(f"  loaded  {label}")
    else:
        print(f"  MISSING {label}  ({fn}) -- run that notebook first")

## 0b · Your personal seed

The notebooks fix the seed to 88 so the "what you should see" blocks are true
on any machine. The report asks for numbers from **your** seed instead, so a
report cannot be copied between groups without the numbers giving it away.

In [ ]:
STUDENT_NUMBER = "20241234"        # <- your AAU study number

SEED = personal_seed(STUDENT_NUMBER)
print("study number :", STUDENT_NUMBER)
print("your seed    :", SEED)

# Your own collocation draw through the slab, and what the exact solution does
# on it.
your_pts = pb.plate_spacetime_points(3000, seed=SEED)
T_you = pb.exact_transient(your_pts[:, 0], your_pts[:, 1], your_pts[:, 2])

print()
print(f"  your mean temperature over the slab : {T_you.mean():.6f}")
print(f"  your peak temperature               : {T_you.max():.6f}")
print(f"  fraction of your points with t < tau: "
      f"{(your_pts[:, 2] < pb.time_constant()).mean() * 100:.1f}%")

## 1 · The evidence, gathered

In [ ]:
if "01 · soft IC" in R and "02 · hard IC" in R:
    s, h = R["01 · soft IC"], R["02 · hard IC"]
    print("SOFT versus HARD initial condition")
    print(error_table(
        [[f"{t:.1f}", f"{a:.3e}", f"{b:.3e}", f"{c:.3e}", f"{d:.3e}"]
         for t, a, b, c, d in zip(s["ts"], s["rel"], h["rel"],
                                  s["abs_err"], h["abs_err"])],
        ["t", "soft rel", "hard rel", "soft abs", "hard abs"]))
    print(f"\n  hard IC, error at t = 0 before training : {float(h['ic_err']):.2e}")
    print(f"  hard IC, edge error before training     : {float(h['edge_err']):.2e}")
    print(f"  final loss, soft / hard : {s['lbfgs'][-1]:.3e} / "
          f"{h['lbfgs'][-1]:.3e}")

    plt.figure(figsize=(7.5, 3.4))
    plt.semilogy(s["ts"], s["abs_err"], "o-", label="soft IC")
    plt.semilogy(h["ts"], h["abs_err"], "s-", label="hard IC")
    plt.xlabel("t"); plt.ylabel("max absolute error")
    plt.legend(frameon=False); plt.grid(alpha=0.25, which="both")
    plt.tight_layout(); plt.show()

if "04 · inverse alpha" in R:
    inv = R["04 · inverse alpha"]
    print(f"\nINVERSE PROBLEM")
    print(f"  true / recovered alpha : {float(inv['alpha_true']):.4f} / "
          f"{float(inv['alpha_hat']):.4f}")
    print(f"  sensor window          : t = {float(inv['sensor_t'][0]):.2f} .. "
          f"{float(inv['sensor_t'][-1]):.2f}"
          f"   ({float(inv['sensor_t'][-1]) / pb.time_constant():.1f} tau)")

## 2 · Your answers

Replace every string. Keep to the word limits.

In [ ]:
# TODO: write your report. Every string below must be replaced.

Q1_EARLY_ERROR = """
(120 words) Why is the error largest at t = 0 with a soft initial condition?
Quote your own numbers from the soft/hard table above, and give at least two
distinct mechanisms -- one about the loss, one about the sampling. (slide 10)
"""

Q2_STEADY_LAST_FRAME = """
(120 words) Why does the steady state appear as the last frame of the
transient? Say what "last" has to mean for that to be true, using the time
constant from notebook 00, and what you would see if you stopped too early.
(slide 16)
"""

Q3_WHY_TRANSIENT = """
(150 words) Why identify alpha from a transient rather than from a steady
state? Answer in terms of what the steady equation does and does not contain.
Then report your recovered alpha and say how much the data supports it.
(slide 20)
"""

Q4_HARMLESS_RELATIVE_ERROR = """
(120 words) Why can a relative error of 100% be harmless at late times? Use
the amplitude column from notebook 00 and your own absolute errors, and state
which of the two numbers you would put in an engineering report, and to whom.
(slide 22)
"""

Q5_WHEN_CLASSICAL = """
(150 words) When does stiffness make you reach for a classical solver instead?
Name the property of the problem that decides it, not the tool.
(slides 21, 24)
"""

Q6_SHORTER_CURVE = """
(150 words) What would happen to the inverse fit if the cooling curve were
shorter -- or, as in the notebook 04 question, moved later? Report what you
actually measured when you moved the sensor window, and give the general rule
for when a parameter is identifiable from data.
"""

NAME = "your name"
GROUP = "your group"

raise NotImplementedError("Write your report, then delete this line")

## 3 · Check, assemble, save

In [ ]:
answers = {
    "1 · Where the error goes": (Q1_EARLY_ERROR, 120),
    "2 · The steady state as the last frame": (Q2_STEADY_LAST_FRAME, 120),
    "3 · Identifying alpha from a transient": (Q3_WHY_TRANSIENT, 150),
    "4 · Relative against absolute": (Q4_HARMLESS_RELATIVE_ERROR, 120),
    "5 · When to use a classical solver": (Q5_WHEN_CLASSICAL, 150),
    "6 · A shorter cooling curve": (Q6_SHORTER_CURVE, 150),
}

problems = []
for title, (text, limit) in answers.items():
    words = len(text.split())
    if text.strip().startswith("(") or f"({limit} words)" in text:
        problems.append(f"{title}: still the prompt")
    elif words > limit * 1.15:
        problems.append(f"{title}: {words} words, limit {limit}")
    elif words < limit * 0.4:
        problems.append(f"{title}: {words} words, too short")

if problems:
    print("Not ready:")
    for p in problems:
        print("  -", p)
else:
    lines = ["# Ex_08.2 — Dynamic heat: a plate, a hole, and time", "",
             f"**{NAME}** · {GROUP}", "",
             f"study number {STUDENT_NUMBER} · seed {SEED}", "",
             "Deep Learning for Engineering · Aalborg University · 2026", "",
             "---", ""]
    for title, (text, _) in answers.items():
        lines += [f"## {title}", "", text.strip(), ""]
    out = os.path.join("Ex08.2_outputs", "Ex08.2_report.md")
    with open(out, "w", encoding="utf-8") as fh:
        fh.write("\n".join(lines))
    print("wrote", out)
    print(f"{sum(len(t.split()) for t, _ in answers.values())} words total")

### The report as a PDF

Moodle shows a PDF inline and a `.md` only as a download, so the cell below
converts the report you just wrote into a PDF (with the figure, if one was
saved) and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex08.2_report.md into Ex08.2_report.pdf, with any figure
# saved as Ex08.2_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md = open("Ex08.2_report.md", encoding="utf-8").read()
figs = sorted(glob.glob("Ex08.2_report*.png"))
if figs:
    imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
    i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
    md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

html = markdown.markdown(md, extensions=["fenced_code", "tables"])
css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                       f"<body>{html}</body></html>", base_url=".").write_pdf("Ex08.2_report.pdf")
print("written Ex08.2_report.pdf", f"with {len(figs)} figure(s)" if figs else "")
try:
    from google.colab import files
    files.download("Ex08.2_report.pdf")
except ImportError:
    pass


## 4 · Extensions

Optional, and each one is a short experiment rather than a new exercise.

- Give the source a duty cycle, `Q(t) = Q0 if (t*4)%1 < 0.55 else 0`, and find
  the peak temperature. Does it exceed the steady value at full load?
- Halve `t_end` and re-run. Which conclusions change?
- Implement causal weighting from slide 11 and compare with hard enforcement.
- Two materials with diffusivities differing by 100×. Where does it break?

## 5 · What Ex_08.2 was for

Three claims you can now defend with your own measurements:

* **Where you put known information decides how much of it survives.** The
  initial condition in the loss is a request the optimiser may refuse; the same
  condition inside the trial solution is a fact it cannot.
* **A transient is not uniformly hard, and one error number hides that.** You
  reported error against time, relative *and* absolute, because at late times
  one of the two is meaningless.
* **An optimiser will always return a parameter.** Whether the data supports it
  is a separate question, and one you have to ask yourself.

From here the geometry and the physics both get harder, and the exact solution
you have been checking against mostly disappears.